# ExoIntel-Prime — Expand `exoplanets.json` with 100 new targets

This Colab notebook reads your current `exoplanets.json` / `data.zip`, checks which planets and light-curve JSON files already exist, queries the NASA Exoplanet Archive for additional transiting targets, searches public TESS/Kepler/K2 light curves through Lightkurve/MAST, phase-folds usable light curves, and writes an updated `data.zip` for your GitHub repo.

Run the cells in order. Start with `DRY_RUN = True` if you want to preview candidates first.

In [ ]:
!pip -q install lightkurve astropy requests tqdm pandas numpy

## 1. Upload your current data

Upload either:
- `exoplanets.json`, or
- `data.zip` containing `data/exoplanets.json` and `data/lightcurves/*.json`.

Uploading both is fine; the direct `exoplanets.json` will override the one inside the ZIP.

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Saving data.zip to data (2).zip
Saving exoplanets.json to exoplanets (1).json
Uploaded: ['data (2).zip', 'exoplanets (1).json']


## 2. Load the update engine

This cell defines all helper functions. It is long, but you only need to run it once.

In [ ]:
"""
ExoIntel-Prime Google Colab cache updater
=========================================

Purpose
-------
Read your existing ExoIntel-Prime data/exoplanets.json and optional data.zip,
query the NASA Exoplanet Archive for additional transiting targets that are NOT
already in your JSON cache, search for public TESS/Kepler/K2 light curves using
Lightkurve, save phase-folded static JSON light curves, merge the new targets
into the cache, and download an updated data.zip.

Recommended Colab usage
-----------------------
1. Upload your current exoplanets.json and optionally data.zip when prompted.
2. Run each section/cell in order.
3. Start with DRY_RUN = True to preview candidates.
4. Then set DRY_RUN = False to actually download light curves and write files.

Notes
-----
- This script does not fabricate fake/scatter light curves.
- A target is added by default only if a compatible public light curve is found.
- Set ADD_TARGETS_WITHOUT_LIGHTCURVE = True if you want catalogue-only targets.
"""

from __future__ import annotations

import json
import math
import os
import re
import shutil
import time
import warnings
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable
from urllib.parse import quote_plus

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

try:
    import lightkurve as lk
except Exception as exc:  # Colab cell should install lightkurve before running.
    raise RuntimeError("lightkurve is not installed. Run: !pip -q install lightkurve astropy") from exc

# ---------------------------------------------------------------------------
# USER SETTINGS
# ---------------------------------------------------------------------------

TARGET_NEW_COUNT = 300
MAX_CANDIDATES_TO_SCAN = 1500
MAX_LIGHTCURVE_ATTEMPTS = 700
ADD_TARGETS_WITHOUT_LIGHTCURVE = False
DRY_RUN = False

PHASE_WINDOW = 0.16
MAX_POINTS_PER_LIGHTCURVE = 1800
MIN_POINTS_IN_TRANSIT_WINDOW = 35
SLEEP_BETWEEN_TARGETS_SEC = 0.6

ROOT = Path("/content/exointel_update")
DATA_DIR = ROOT / "data"
LIGHTCURVE_DIR = DATA_DIR / "lightcurves"
EXOPLANETS_PATH = DATA_DIR / "exoplanets.json"
OUTPUT_ZIP = Path("/content/exointel_prime_updated_data.zip")

TAP_URL = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"

CORE_COLUMNS = [
    "pl_name",
    "hostname",
    "sy_snum",
    "sy_pnum",
    "ra",
    "dec",
    "pl_orbper",
    "pl_orbsmax",
    "pl_ratror",
    "pl_rade",
    "pl_bmasse",
    "pl_orbincl",
    "pl_orbeccen",
    "pl_trandep",
    "pl_trandur",
    "pl_tranmid",
    "st_teff",
    "st_rad",
    "st_mass",
    "st_logg",
    "st_met",
    "disc_year",
    "discoverymethod",
    "disc_facility",
]

REQUIRED_FIELDS = [
    "pl_name",
    "hostname",
    "pl_orbper",
    "pl_ratror",
    "pl_orbincl",
    "pl_trandep",
    "st_rad",
    "st_teff",
]

# We deliberately pull from a wide temperature range so the WebGL star colour
# has meaningful M/K/G/F/A-type examples. The weights are approximate quotas.
TEMPERATURE_BUCKETS = [
    {"label": "very_cool_m", "min": 2400, "max": 3700, "quota": 28},
    {"label": "cool_k", "min": 3700, "max": 5200, "quota": 22},
    {"label": "solar_g", "min": 5200, "max": 6000, "quota": 16},
    {"label": "warm_f", "min": 6000, "max": 7500, "quota": 20},
    {"label": "hot_a_b", "min": 7500, "max": 20000, "quota": 14},
]

# ---------------------------------------------------------------------------
# GENERIC HELPERS
# ---------------------------------------------------------------------------

def slugify(value: Any) -> str:
    text = str(value or "unknown-target").strip().lower()
    text = text.replace("+", " plus ")
    text = re.sub(r"[’'\"]", "", text)
    text = re.sub(r"[^a-z0-9]+", "-", text)
    text = re.sub(r"^-+|-+$", "", text)
    return text or "unknown-target"


def as_float(value: Any, fallback: float | None = None) -> float | None:
    if value is None or value == "":
        return fallback
    try:
        number = float(value)
    except Exception:
        return fallback
    return number if math.isfinite(number) else fallback


def as_int(value: Any, fallback: int | None = None) -> int | None:
    number = as_float(value, None)
    return int(number) if number is not None else fallback


def clean_text(value: Any) -> str | None:
    if value is None:
        return None
    text = str(value).strip()
    return text if text else None


def normalise_name(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "").strip().lower())


def finite_array(values: Iterable[Any]) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    return arr[np.isfinite(arr)]


def safe_percentile(values: np.ndarray, pct: float, fallback: float = np.nan) -> float:
    clean = finite_array(values)
    if clean.size == 0:
        return fallback
    return float(np.nanpercentile(clean, pct))


def normalise_existing_payload(payload: Any) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    """Return metadata dict + target list whether cache is dict or list."""
    if isinstance(payload, dict):
        targets = payload.get("targets", [])
        if not isinstance(targets, list):
            raise ValueError("exoplanets.json dict exists, but 'targets' is not a list")
        meta = {k: v for k, v in payload.items() if k != "targets"}
        return meta, targets
    if isinstance(payload, list):
        return {}, payload
    raise ValueError("Unsupported exoplanets.json format. Expected dict or list.")


def target_key(row: dict[str, Any]) -> str:
    return normalise_name(row.get("pl_name"))


def host_key(row: dict[str, Any]) -> str:
    return normalise_name(row.get("hostname"))


def required_complete(row: dict[str, Any]) -> bool:
    return all(row.get(k) is not None for k in REQUIRED_FIELDS)


def row_to_cache_target(row: dict[str, Any]) -> dict[str, Any]:
    pl_name = clean_text(row.get("pl_name"))
    trandep = as_float(row.get("pl_trandep"), None)
    out = {
        "pl_name": pl_name,
        "hostname": clean_text(row.get("hostname")),
        "sy_snum": as_int(row.get("sy_snum"), None),
        "sy_pnum": as_int(row.get("sy_pnum"), None),
        "ra": as_float(row.get("ra"), None),
        "dec": as_float(row.get("dec"), None),
        "pl_orbper": as_float(row.get("pl_orbper"), None),
        "pl_orbsmax": as_float(row.get("pl_orbsmax"), None),
        "pl_ratror": as_float(row.get("pl_ratror"), None),
        "pl_rade": as_float(row.get("pl_rade"), None),
        "pl_bmasse": as_float(row.get("pl_bmasse"), None),
        "pl_orbincl": as_float(row.get("pl_orbincl"), None),
        "pl_orbeccen": as_float(row.get("pl_orbeccen"), 0.0),
        "pl_trandep": trandep,
        "pl_trandep_percent": (trandep / 10000.0) if trandep is not None else None,
        "pl_trandur": as_float(row.get("pl_trandur"), None),
        "pl_tranmid": as_float(row.get("pl_tranmid"), None),
        "st_teff": as_float(row.get("st_teff"), None),
        "st_rad": as_float(row.get("st_rad"), None),
        "st_mass": as_float(row.get("st_mass"), None),
        "st_logg": as_float(row.get("st_logg"), None),
        "st_met": as_float(row.get("st_met"), None),
        "disc_year": as_int(row.get("disc_year"), None),
        "discoverymethod": clean_text(row.get("discoverymethod")) or "Transit",
        "disc_facility": clean_text(row.get("disc_facility")),
        "lightcurve_file": f"{slugify(pl_name)}.json" if pl_name else None,
        "lightcurve_available": False,
    }
    return out

# ---------------------------------------------------------------------------
# STEP 1: WORKSPACE AND INPUT FILES
# ---------------------------------------------------------------------------

def prepare_workspace(uploaded_files: dict[str, bytes] | None = None) -> None:
    """Prepare /content/exointel_update and copy uploaded files if provided.

    In Colab you can pass google.colab.files.upload() result.
    This function accepts either:
    - exoplanets.json directly
    - data.zip containing exoplanets.json and lightcurves/
    """
    if ROOT.exists():
        shutil.rmtree(ROOT)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    LIGHTCURVE_DIR.mkdir(parents=True, exist_ok=True)

    uploaded_files = uploaded_files or {}

    # Write uploaded files into /content first.
    for name, blob in uploaded_files.items():
        p = Path("/content") / name
        p.write_bytes(blob)

    # If data.zip exists, unzip first.
    data_zip = Path("/content/data.zip")
    if data_zip.exists():
        with zipfile.ZipFile(data_zip, "r") as zf:
            zf.extractall(DATA_DIR)
        # Some zips contain data/exoplanets.json, some contain exoplanets.json at root.
        nested = DATA_DIR / "data"
        if nested.exists():
            if (nested / "exoplanets.json").exists():
                shutil.copy2(nested / "exoplanets.json", EXOPLANETS_PATH)
            if (nested / "lightcurves").exists():
                for p in (nested / "lightcurves").glob("*.json"):
                    shutil.copy2(p, LIGHTCURVE_DIR / p.name)

    # Direct uploaded exoplanets.json overrides zip cache.
    direct_json = Path("/content/exoplanets.json")
    if direct_json.exists():
        shutil.copy2(direct_json, EXOPLANETS_PATH)

    if not EXOPLANETS_PATH.exists():
        raise FileNotFoundError("Upload exoplanets.json or data.zip first.")

    # Ensure lightcurves folder exists even if missing.
    LIGHTCURVE_DIR.mkdir(parents=True, exist_ok=True)


def load_existing_cache() -> tuple[dict[str, Any], list[dict[str, Any]]]:
    with EXOPLANETS_PATH.open("r", encoding="utf-8") as f:
        payload = json.load(f)
    return normalise_existing_payload(payload)


def inspect_existing_cache() -> tuple[dict[str, Any], list[dict[str, Any]], set[str], set[str]]:
    meta, targets = load_existing_cache()
    existing_names = {target_key(t) for t in targets if target_key(t)}
    existing_files = {p.name.lower() for p in LIGHTCURVE_DIR.glob("*.json")}
    json_lc_files = {str(t.get("lightcurve_file") or "").lower() for t in targets if t.get("lightcurve_file")}
    existing_lc_files = existing_files | json_lc_files

    print("Existing cache inspection")
    print("-------------------------")
    print(f"Targets in JSON:        {len(targets)}")
    print(f"Lightcurve JSON files:  {len(existing_files)}")
    print(f"Targets with LC flag:   {sum(bool(t.get('lightcurve_available')) for t in targets)}")
    print(f"Existing target names:  {len(existing_names)} unique")
    print(f"Schema:                 {meta.get('schema', 'list-format or unknown')}")
    return meta, targets, existing_names, existing_lc_files

# ---------------------------------------------------------------------------
# STEP 2: NASA EXOPLANET ARCHIVE QUERY
# ---------------------------------------------------------------------------

def tap_query(adql: str, timeout: int = 90) -> pd.DataFrame:
    payload = {
        "query": adql,
        "format": "json",
    }
    response = requests.get(TAP_URL, params=payload, timeout=timeout)
    response.raise_for_status()
    data = response.json()
    if not isinstance(data, list):
        raise ValueError("NASA Exoplanet Archive TAP did not return a JSON row list")
    return pd.DataFrame(data)


def make_adql(limit: int, teff_min: float, teff_max: float, prefer_tess: bool = True) -> str:
    columns = ",\n  ".join(CORE_COLUMNS)
    tess_clause = ""
    if prefer_tess:
        tess_clause = """
  AND (
       LOWER(COALESCE(disc_facility, '')) LIKE '%tess%'
       OR LOWER(COALESCE(disc_facility, '')) LIKE '%transiting exoplanet survey satellite%'
       OR LOWER(pl_name) LIKE 'toi-%'
      )
"""
    return f"""
SELECT TOP {int(limit)}
  {columns}
FROM pscomppars
WHERE tran_flag = 1
  AND pl_name IS NOT NULL
  AND hostname IS NOT NULL
  AND pl_orbper IS NOT NULL
  AND pl_ratror IS NOT NULL
  AND pl_orbincl IS NOT NULL
  AND pl_trandep IS NOT NULL
  AND st_rad IS NOT NULL
  AND st_teff IS NOT NULL
  AND st_teff >= {float(teff_min)}
  AND st_teff < {float(teff_max)}
  {tess_clause}
ORDER BY pl_trandep DESC
"""


def fetch_candidate_targets(existing_names: set[str]) -> list[dict[str, Any]]:
    all_rows: list[dict[str, Any]] = []
    seen = set(existing_names)

    print("Querying NASA Exoplanet Archive by stellar temperature bucket...")
    for bucket in TEMPERATURE_BUCKETS:
        limit = max(80, bucket["quota"] * 5)
        for prefer_tess in (True, False):
            try:
                df = tap_query(make_adql(limit, bucket["min"], bucket["max"], prefer_tess=prefer_tess))
            except Exception as exc:
                print(f"  TAP query failed for {bucket['label']} prefer_tess={prefer_tess}: {exc}")
                continue

            bucket_added = 0
            for _, raw in df.iterrows():
                row = row_to_cache_target(raw.to_dict())
                key = target_key(row)
                if not key or key in seen:
                    continue
                if not required_complete(row):
                    continue
                row["selection_bucket"] = bucket["label"]
                row["selection_prefer_tess"] = bool(prefer_tess)
                seen.add(key)
                all_rows.append(row)
                bucket_added += 1

            print(f"  {bucket['label']:<13} prefer_tess={prefer_tess:<5} fetched new unique: {bucket_added}")
            # If preferred TESS gave enough rows for this bucket, still run general if total candidates are low.

    # Prioritise TESS-ish rows, then temperature diversity, then depth.
    def score(row: dict[str, Any]) -> tuple[float, float, float]:
        facility = str(row.get("disc_facility") or "").lower()
        name = str(row.get("pl_name") or "").lower()
        tess_bonus = 1.0 if ("tess" in facility or name.startswith("toi-")) else 0.0
        teff = as_float(row.get("st_teff"), 5778) or 5778
        # prefer extremes slightly, because user wants cooler/hotter systems too
        temp_extreme = abs(teff - 5600.0) / 5600.0
        depth = as_float(row.get("pl_trandep"), 0.0) or 0.0
        return (tess_bonus, temp_extreme, depth)

    all_rows.sort(key=score, reverse=True)
    print(f"Total unique candidate targets not in current JSON: {len(all_rows)}")
    return all_rows[:MAX_CANDIDATES_TO_SCAN]

# ---------------------------------------------------------------------------
# STEP 3: LIGHTKURVE SEARCH + PHASE-FOLDED JSON EXPORT
# ---------------------------------------------------------------------------

def candidate_search_terms(row: dict[str, Any]) -> list[str]:
    terms = []
    for value in [row.get("hostname"), row.get("pl_name")]:
        if value:
            terms.append(str(value))
    # Remove duplicates while preserving order.
    out = []
    seen = set()
    for t in terms:
        clean = t.strip()
        if clean and clean.lower() not in seen:
            out.append(clean)
            seen.add(clean.lower())
    return out


def rank_search_result_table(search_result: Any) -> list[int]:
    """Return indices ranked by mission/author/cadence preference."""
    try:
        table = search_result.table
    except Exception:
        return []

    rows = []
    for i, row in enumerate(table):
        mission = str(row.get("mission", "")).lower()
        author = str(row.get("author", "")).lower()
        exptime = as_float(row.get("exptime"), None)
        score = 0.0

        if "tess" in mission:
            score += 60
        elif "kepler" in mission:
            score += 50
        elif "k2" in mission:
            score += 45

        if author in {"spoc", "kepler", "k2"}:
            score += 28
        if author in {"qlp", "tess-sffi", "eleanor", "everest", "k2sff"}:
            score += 20
        if exptime is not None:
            # prefer 2-min/20-sec/30-min products before very unusual products
            if exptime <= 1800:
                score += 12
            else:
                score += 4

        rows.append((score, i))

    rows.sort(reverse=True)
    return [i for score, i in rows]


def search_and_download_lightcurve(row: dict[str, Any]) -> Any | None:
    terms = candidate_search_terms(row)

    for term in terms:
        for mission in (["TESS"], ["Kepler"], ["K2"], None):
            try:
                sr = lk.search_lightcurve(term, mission=mission) if mission else lk.search_lightcurve(term)
            except Exception:
                continue

            if len(sr) == 0:
                continue

            ranked_indices = rank_search_result_table(sr)
            for idx in ranked_indices[:8]:
                try:
                    lc = sr[idx].download(quality_bitmask="default")
                    if lc is not None and len(lc) > 20:
                        return lc
                except Exception:
                    continue

    return None


def extract_time_jd(lc: Any) -> np.ndarray:
    try:
        return np.asarray(lc.time.jd, dtype=float)
    except Exception:
        try:
            return np.asarray(lc.time.value, dtype=float)
        except Exception:
            return np.asarray([], dtype=float)


def extract_flux_values(lc: Any) -> np.ndarray:
    # lightkurve normalize() usually returns dimensionless flux near unity.
    try:
        norm = lc.remove_nans().normalize(unit="unscaled")
    except Exception:
        try:
            norm = lc.remove_nans().normalize()
        except Exception:
            norm = lc

    for attr in ["pdcsap_flux", "sap_flux", "flux"]:
        try:
            values = getattr(norm, attr)
            arr = np.asarray(values.value if hasattr(values, "value") else values, dtype=float)
            if np.isfinite(arr).sum() > 20:
                med = np.nanmedian(arr)
                if np.isfinite(med) and med != 0:
                    # If not near unity, normalize manually.
                    if not (0.5 < med < 1.5):
                        arr = arr / med
                return arr
        except Exception:
            continue
    return np.asarray([], dtype=float)


def phase_fold_lightcurve(lc: Any, row: dict[str, Any]) -> dict[str, Any] | None:
    period = as_float(row.get("pl_orbper"), None)
    t0 = as_float(row.get("pl_tranmid"), None)
    if period is None or period <= 0:
        return None

    time_jd = extract_time_jd(lc)
    flux = extract_flux_values(lc)

    n = min(len(time_jd), len(flux))
    if n < 50:
        return None
    time_jd = time_jd[:n]
    flux = flux[:n]

    valid = np.isfinite(time_jd) & np.isfinite(flux) & (flux > 0)
    time_jd = time_jd[valid]
    flux = flux[valid]
    if len(time_jd) < 50:
        return None

    # Robust normalization in case lightkurve did not fully normalize.
    med_flux = np.nanmedian(flux)
    if np.isfinite(med_flux) and med_flux != 0:
        flux = flux / med_flux

    # Try JD T0 first. If mismatch with space mission time systems is bad,
    # also try common offsets and keep the one with the deepest transit near phase 0.
    t0_candidates = []
    if t0 is not None:
        t0_candidates.extend([t0, t0 - 2457000.0, t0 - 2454833.0, t0 - 2400000.5])
    else:
        t0_candidates.append(time_jd[0])

    best = None
    for t0_try in t0_candidates:
        phase = ((time_jd - t0_try + 0.5 * period) % period) / period - 0.5
        window = np.abs(phase) <= PHASE_WINDOW
        if window.sum() < MIN_POINTS_IN_TRANSIT_WINDOW:
            continue

        phase_w = phase[window]
        flux_w = flux[window]

        # Remove extreme outliers; keep real scatter.
        lo = safe_percentile(flux_w, 0.5, np.nan)
        hi = safe_percentile(flux_w, 99.5, np.nan)
        keep = np.isfinite(phase_w) & np.isfinite(flux_w)
        if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
            keep &= (flux_w >= lo) & (flux_w <= hi)
        phase_w = phase_w[keep]
        flux_w = flux_w[keep]

        if len(phase_w) < MIN_POINTS_IN_TRANSIT_WINDOW:
            continue

        # Downsample deterministically for frontend size.
        order = np.argsort(phase_w)
        phase_w = phase_w[order]
        flux_w = flux_w[order]
        if len(phase_w) > MAX_POINTS_PER_LIGHTCURVE:
            idx = np.linspace(0, len(phase_w) - 1, MAX_POINTS_PER_LIGHTCURVE).astype(int)
            phase_w = phase_w[idx]
            flux_w = flux_w[idx]

        central = np.abs(phase_w) <= min(0.035, PHASE_WINDOW / 2)
        oot = np.abs(phase_w) >= PHASE_WINDOW * 0.65
        central_depth = 1.0 - np.nanmedian(flux_w[central]) if central.sum() >= 5 else 0.0
        oot_scatter = np.nanstd(flux_w[oot]) if oot.sum() >= 5 else np.nanstd(flux_w)
        score = central_depth / max(oot_scatter, 1e-6)

        result = {
            "phase": phase_w.astype(float),
            "flux": flux_w.astype(float),
            "score": float(score),
            "points": int(len(phase_w)),
            "t0_used": float(t0_try),
        }
        if best is None or result["score"] > best["score"]:
            best = result

    if best is None:
        return None

    # Estimate uncertainty as local OOT scatter; frontend supports optional error.
    phase = best["phase"]
    flux = best["flux"]
    oot = np.abs(phase) >= PHASE_WINDOW * 0.65
    err_value = float(np.nanstd(flux[oot])) if oot.sum() >= 10 else float(np.nanstd(flux))
    if not np.isfinite(err_value) or err_value <= 0:
        err_value = 0.0005
    error = np.full_like(flux, err_value, dtype=float)

    return {
        "source": "MAST public light curve via Lightkurve; phase-folded for ExoIntel-Prime",
        "target": row.get("pl_name"),
        "host": row.get("hostname"),
        "period_days": period,
        "t0_used": best["t0_used"],
        "phase_window": PHASE_WINDOW,
        "points": best["points"],
        "phase": [round(float(x), 8) for x in phase],
        "flux": [round(float(x), 8) for x in flux],
        "error": [round(float(x), 8) for x in error],
    }


def save_lightcurve_json(row: dict[str, Any], lc_payload: dict[str, Any]) -> str:
    filename = f"{slugify(row.get('pl_name'))}.json"
    path = LIGHTCURVE_DIR / filename
    with path.open("w", encoding="utf-8") as f:
        json.dump(lc_payload, f, indent=2)
    return filename

# ---------------------------------------------------------------------------
# STEP 4: MERGE NEW TARGETS
# ---------------------------------------------------------------------------

def build_updated_cache(meta: dict[str, Any], existing_targets: list[dict[str, Any]], new_targets: list[dict[str, Any]]) -> dict[str, Any]:
    merged = []
    seen = set()

    for row in list(existing_targets) + list(new_targets):
        key = target_key(row)
        if not key or key in seen:
            continue
        seen.add(key)
        merged.append(row)

    # Keep original ordering first, then new rows; optionally sort new rows by temperature/depth not necessary.
    lightcurve_count = sum(bool(t.get("lightcurve_available")) for t in merged)

    columns = list(meta.get("columns") or [])
    for col in [
        "pl_name", "hostname", "sy_snum", "sy_pnum", "ra", "dec",
        "pl_orbper", "pl_orbsmax", "pl_ratror", "pl_rade", "pl_bmasse",
        "pl_orbincl", "pl_orbeccen", "pl_trandep", "pl_trandep_percent",
        "pl_trandur", "pl_tranmid", "st_teff", "st_rad", "st_mass",
        "st_logg", "st_met", "disc_year", "discoverymethod", "disc_facility",
        "lightcurve_file", "lightcurve_available"
    ]:
        if col not in columns:
            columns.append(col)

    return {
        "schema": "exointel-prime-expanded-cache-v3-colab",
        "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "source": "Merged previous ExoIntel-Prime cache with additional NASA Exoplanet Archive targets and Lightkurve/MAST public light curves",
        "target_count": len(merged),
        "lightcurve_count": lightcurve_count,
        "lightcurve_directory": "data/lightcurves",
        "columns": columns,
        "targets": merged,
    }


def update_cache_with_new_targets(candidates: list[dict[str, Any]], meta: dict[str, Any], existing_targets: list[dict[str, Any]], existing_lc_files: set[str]) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    added: list[dict[str, Any]] = []
    attempted = 0

    print(f"Scanning up to {min(len(candidates), MAX_CANDIDATES_TO_SCAN)} candidates.")
    print(f"Goal: add {TARGET_NEW_COUNT} new targets. ADD_TARGETS_WITHOUT_LIGHTCURVE={ADD_TARGETS_WITHOUT_LIGHTCURVE}")

    for row in tqdm(candidates[:MAX_CANDIDATES_TO_SCAN], desc="Candidates"):
        if len(added) >= TARGET_NEW_COUNT:
            break
        if attempted >= MAX_LIGHTCURVE_ATTEMPTS and not ADD_TARGETS_WITHOUT_LIGHTCURVE:
            print("Reached MAX_LIGHTCURVE_ATTEMPTS before target goal. Increase setting if needed.")
            break

        filename = f"{slugify(row.get('pl_name'))}.json"
        if filename.lower() in existing_lc_files:
            # Existing LC file found even if target was not in JSON; mark available and add.
            row["lightcurve_file"] = filename
            row["lightcurve_available"] = True
            added.append(row)
            continue

        if DRY_RUN:
            row["lightcurve_available"] = False
            row["lightcurve_file"] = filename
            added.append(row)
            continue

        attempted += 1
        lc = search_and_download_lightcurve(row)
        lc_payload = None
        if lc is not None:
            lc_payload = phase_fold_lightcurve(lc, row)

        if lc_payload is not None:
            saved = save_lightcurve_json(row, lc_payload)
            row["lightcurve_file"] = saved
            row["lightcurve_available"] = True
            added.append(row)
            existing_lc_files.add(saved.lower())
            print(f"  Added with LC: {row.get('pl_name')} -> {saved} ({lc_payload['points']} pts)")
        elif ADD_TARGETS_WITHOUT_LIGHTCURVE:
            row["lightcurve_file"] = filename
            row["lightcurve_available"] = False
            added.append(row)
            print(f"  Added catalogue-only: {row.get('pl_name')}")
        else:
            print(f"  Skipped no usable LC: {row.get('pl_name')}")

        time.sleep(SLEEP_BETWEEN_TARGETS_SEC)

    updated = build_updated_cache(meta, existing_targets, added)
    with EXOPLANETS_PATH.open("w", encoding="utf-8") as f:
        json.dump(updated, f, indent=2)

    return updated, added

# ---------------------------------------------------------------------------
# STEP 5: VALIDATE + ZIP + DOWNLOAD
# ---------------------------------------------------------------------------

def validate_cache(cache: dict[str, Any]) -> pd.DataFrame:
    rows = cache.get("targets", [])
    report = []
    for row in rows:
        fname = row.get("lightcurve_file")
        expected = LIGHTCURVE_DIR / str(fname) if fname else None
        report.append({
            "pl_name": row.get("pl_name"),
            "hostname": row.get("hostname"),
            "st_teff": row.get("st_teff"),
            "pl_trandep": row.get("pl_trandep"),
            "lightcurve_available": bool(row.get("lightcurve_available")),
            "lightcurve_file": fname,
            "file_exists": bool(expected and expected.exists()),
        })
    df = pd.DataFrame(report)
    print("Updated cache validation")
    print("------------------------")
    print(f"Targets:              {len(df)}")
    print(f"LC flag true:         {df['lightcurve_available'].sum()}")
    print(f"LC files existing:    {df['file_exists'].sum()}")
    print(f"Teff range:           {df['st_teff'].min()} K -> {df['st_teff'].max()} K")
    print(f"Lightcurve JSON dir:  {LIGHTCURVE_DIR}")
    print(f"Output JSON:          {EXOPLANETS_PATH}")
    return df


def make_output_zip() -> Path:
    if OUTPUT_ZIP.exists():
        OUTPUT_ZIP.unlink()
    with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(EXOPLANETS_PATH, "data/exoplanets.json")
        for p in sorted(LIGHTCURVE_DIR.glob("*.json")):
            zf.write(p, f"data/lightcurves/{p.name}")
    print(f"Created: {OUTPUT_ZIP}")
    print(f"Size: {OUTPUT_ZIP.stat().st_size / 1_000_000:.2f} MB")
    return OUTPUT_ZIP

# ---------------------------------------------------------------------------
# STEP 6: ONE-CALL RUNNER FOR AFTER YOU TEST CELLS
# ---------------------------------------------------------------------------

def run_full_update(uploaded_files: dict[str, bytes] | None = None) -> tuple[dict[str, Any], list[dict[str, Any]], Path]:
    prepare_workspace(uploaded_files)
    meta, existing_targets, existing_names, existing_lc_files = inspect_existing_cache()
    candidates = fetch_candidate_targets(existing_names)
    updated, added = update_cache_with_new_targets(candidates, meta, existing_targets, existing_lc_files)
    validate_cache(updated)
    zip_path = make_output_zip()
    print("\nDone.")
    print(f"New targets added: {len(added)}")
    print(f"Updated target count: {updated['target_count']}")
    print(f"Updated LC count: {updated['lightcurve_count']}")
    return updated, added, zip_path

## 3. Settings

For a first test, keep `DRY_RUN = True`. When you are ready to actually download light curves, set `DRY_RUN = False` and rerun from this cell onward.

In [ ]:
TARGET_NEW_COUNT = 100
MAX_CANDIDATES_TO_SCAN = 450
MAX_LIGHTCURVE_ATTEMPTS = 170
ADD_TARGETS_WITHOUT_LIGHTCURVE = False
DRY_RUN = True   # change to False when you want to download and write real LC JSON files

PHASE_WINDOW = 0.16
MAX_POINTS_PER_LIGHTCURVE = 1800
MIN_POINTS_IN_TRANSIT_WINDOW = 35

## 4. Inspect your existing JSON and light curves

In [ ]:
prepare_workspace(uploaded)
meta, existing_targets, existing_names, existing_lc_files = inspect_existing_cache()

Existing cache inspection
-------------------------
Targets in JSON:        200
Lightcurve JSON files:  174
Targets with LC flag:   174
Existing target names:  200 unique
Schema:                 exointel-prime-final-merged-cache-v2-cleaned


## 5. Query candidates not already in your JSON

The query is temperature-bucketed so the new archive does not only contain Sun-like stars. It tries to include cooler M/K hosts and hotter F/A hosts as well.

In [ ]:
candidates = fetch_candidate_targets(existing_names)
preview_cols = ["pl_name", "hostname", "st_teff", "pl_trandep", "pl_ratror", "disc_facility", "selection_bucket", "selection_prefer_tess"]
pd.DataFrame(candidates)[preview_cols].head(30)

Querying NASA Exoplanet Archive by stellar temperature bucket...
  very_cool_m   prefer_tess=1     fetched new unique: 126
  very_cool_m   prefer_tess=0     fetched new unique: 44
  cool_k        prefer_tess=1     fetched new unique: 109
  cool_k        prefer_tess=0     fetched new unique: 67
  solar_g       prefer_tess=1     fetched new unique: 79
  solar_g       prefer_tess=0     fetched new unique: 0
  warm_f        prefer_tess=1     fetched new unique: 98
  warm_f        prefer_tess=0     fetched new unique: 51
  hot_a_b       prefer_tess=1     fetched new unique: 3
  hot_a_b       prefer_tess=0     fetched new unique: 11
Total unique candidate targets not in current JSON: 588


,pl_name,hostname,st_teff,pl_trandep,pl_ratror,disc_facility,selection_bucket,selection_prefer_tess
0,LP 791-18 c,LP 791-18,2960.0,1.893141,0.12518,Transiting Exoplanet Survey Satellite (TESS),very_cool_m,True
1,LP 791-18 b,LP 791-18,2960.0,0.420053,0.06100,Transiting Exoplanet Survey Satellite (TESS),very_cool_m,True
2,TOI-6894 b,TOI-6894,3007.0,16.658991,0.38600,Transiting Exoplanet Survey Satellite (TESS),very_cool_m,True
3,TOI-2267 d,TOI-2267 B,3022.0,0.114210,0.12100,Transiting Exoplanet Survey Satellite (TESS),very_cool_m,True
4,TOI-2267 c,TOI-2267 A,3030.0,0.282000,0.05040,Transiting Exoplanet Survey Satellite (TESS),very_cool_m,True
5,TOI-2267 b,TOI-2267 A,3030.0,0.223000,0.04410,Transiting Exoplanet Survey Satellite (TESS),very_cool_m,True
6,TOI-1080 b,TOI-1080,3065.0,0.368075,0.05480,Transiting Exoplanet Survey Satellite (TESS),very_cool_m,True
7,TOI-1227 b,TOI-1227,3072.0,2.069067,0.15680,Transiting Exoplanet Survey Satellite (TESS),very_cool_m,True
8,TOI-715 b,TOI-715,3075.0,0.448000,0.06180,Transiting Exoplanet Survey Satellite (TESS),very_cool_m,True
9,TOI-6008 b,TOI-6008,3075.0,0.151700,0.03867,Transiting Exoplanet Survey Satellite (TESS),very_cool_m,True


## 6. Dry run merge preview

This does **not** download light curves if `DRY_RUN=True`. It shows what would be added.

In [ ]:
updated_preview, added_preview = update_cache_with_new_targets(
    candidates,
    meta,
    existing_targets,
    existing_lc_files
)
print("Preview added:", len(added_preview))
pd.DataFrame(added_preview)[["pl_name", "hostname", "st_teff", "pl_trandep", "selection_bucket", "lightcurve_available"]].head(30)

Scanning up to 588 candidates.
Goal: add 300 new targets. ADD_TARGETS_WITHOUT_LIGHTCURVE=False


Candidates:   0%|          | 0/588 [00:00<?, ?it/s]

  Added with LC: LP 791-18 c -> lp-791-18-c.json (1800 pts)
  Added with LC: LP 791-18 b -> lp-791-18-b.json (1800 pts)
  Added with LC: TOI-6894 b -> toi-6894-b.json (1800 pts)
  Added with LC: TOI-2267 d -> toi-2267-d.json (1800 pts)


Could not resolve "TOI-2267 A" to a sky position.
ERROR:lightkurve.search:Could not resolve "TOI-2267 A" to a sky position.
Could not resolve "TOI-2267 A" to a sky position.
ERROR:lightkurve.search:Could not resolve "TOI-2267 A" to a sky position.
Could not resolve "TOI-2267 A" to a sky position.
ERROR:lightkurve.search:Could not resolve "TOI-2267 A" to a sky position.
Could not resolve "TOI-2267 A" to a sky position.
ERROR:lightkurve.search:Could not resolve "TOI-2267 A" to a sky position.


  Added with LC: TOI-2267 c -> toi-2267-c.json (1800 pts)
  Added with LC: TOI-2267 b -> toi-2267-b.json (1800 pts)
  Added with LC: TOI-1080 b -> toi-1080-b.json (1800 pts)
  Added with LC: TOI-1227 b -> toi-1227-b.json (1800 pts)
  Added with LC: TOI-715 b -> toi-715-b.json (1800 pts)
  Added with LC: TOI-6008 b -> toi-6008-b.json (1800 pts)
  Added with LC: LHS 3844 b -> lhs-3844-b.json (1800 pts)
  Added with LC: TOI-7166 b -> toi-7166-b.json (1800 pts)
  Added with LC: TOI-6716 b -> toi-6716-b.json (1800 pts)
  Added with LC: TOI-2120 b -> toi-2120-b.json (1800 pts)
  Added with LC: TOI-3884 b -> toi-3884-b.json (1800 pts)
  Added with LC: TOI-7384 b -> toi-7384-b.json (1800 pts)
  Added with LC: TOI-1696 b -> toi-1696-b.json (1800 pts)
  Added with LC: TOI-2266 b -> toi-2266-b.json (1800 pts)
  Added with LC: TOI-6086 b -> toi-6086-b.json (1800 pts)
  Added with LC: TOI-540 b -> toi-540-b.json (1800 pts)
  Added with LC: TOI-1680 b -> toi-1680-b.json (1800 pts)


...

  Added with LC: TOI-1259 A b -> toi-1259-a-b.json (1800 pts)
  Added with LC: TOI-201 b -> toi-201-b.json (1800 pts)
  Added with LC: TIC 147027702 b -> tic-147027702-b.json (1800 pts)
  Added with LC: TOI-5380 b -> toi-5380-b.json (1800 pts)
  Added with LC: TOI-622 b -> toi-622-b.json (1800 pts)
  Added with LC: TOI-1836 b -> toi-1836-b.json (1800 pts)
  Added with LC: TOI-1301.01 -> toi-1301-01.json (1800 pts)
  Added with LC: HR 858 b -> hr-858-b.json (1800 pts)
  Added with LC: HR 858 d -> hr-858-d.json (1800 pts)
  Added with LC: HR 858 c -> hr-858-c.json (1800 pts)
  Added with LC: TOI-431 d -> toi-431-d.json (1800 pts)
  Added with LC: TOI-431 b -> toi-431-b.json (1800 pts)
  Added with LC: TOI-2104 b -> toi-2104-b.json (1800 pts)
  Added with LC: TOI-1859 b -> toi-1859-b.json (1800 pts)
  Added with LC: TOI-815 c -> toi-815-c.json (1800 pts)
  Added with LC: TOI-1135 b -> toi-1135-b.json (1800 pts)


  Added with LC: TOI-1416 b -> toi-1416-b.json (1800 pts)
  Added with LC: TOI-1107 b -> toi-1107-b.json (1800 pts)
  Added with LC: HD 152843 c -> hd-152843-c.json (1800 pts)
  Added with LC: HD 152843 b -> hd-152843-b.json (1800 pts)
  Added with LC: TOI-4551 b -> toi-4551-b.json (1800 pts)
  Added with LC: TOI-205 b -> toi-205-b.json (1800 pts)
  Added with LC: TOI-5153 b -> toi-5153-b.json (1800 pts)
  Added with LC: TOI-677 b -> toi-677-b.json (1800 pts)
  Added with LC: TOI-6707 b -> toi-6707-b.json (1800 pts)


  Added with LC: TOI-1807 b -> toi-1807-b.json (1800 pts)
  Added with LC: TOI-2154 b -> toi-2154-b.json (1800 pts)
  Added with LC: TOI-1333 b -> toi-1333-b.json (1800 pts)
  Added with LC: HIP 113103 b -> hip-113103-b.json (1800 pts)
  Added with LC: TOI-4603 b -> toi-4603-b.json (1800 pts)
  Added with LC: TOI-892 b -> toi-892-b.json (1800 pts)
  Added with LC: TOI-4507 b -> toi-4507-b.json (1800 pts)
  Added with LC: TOI-150.01 -> toi-150-01.json (1800 pts)
  Added with LC: TOI-2046 b -> toi-2046-b.json (1800 pts)
  Added with LC: TOI-628 b -> toi-628-b.json (1800 pts)
  Added with LC: TOI-1898 b -> toi-1898-b.json (1800 pts)
  Added with LC: TOI-333 b -> toi-333-b.json (1800 pts)
  Added with LC: TOI-4010 d -> toi-4010-d.json (1800 pts)
  Added with LC: TOI-5301 b -> toi-5301-b.json (1800 pts)
  Added with LC: TOI-5786 b -> toi-5786-b.json (1800 pts)
  Added with LC: TOI-3568 b -> toi-3568-b.json (1800 pts)
  Added with LC: TOI-4377 b -> toi-4377-b.json (1800 pts)
  Added with LC:

  Added with LC: TOI-5599 b -> toi-5599-b.json (1800 pts)
  Added with LC: TOI-3261 b -> toi-3261-b.json (1800 pts)
  Added with LC: TIC 245076932 b -> tic-245076932-b.json (1800 pts)
  Added with LC: HD 332231 b -> hd-332231-b.json (1800 pts)
  Added with LC: TOI-808 b -> toi-808-b.json (1800 pts)
  Added with LC: HD 221416 b -> hd-221416-b.json (1800 pts)
Preview added: 300


,pl_name,hostname,st_teff,pl_trandep,selection_bucket,lightcurve_available
0,LP 791-18 c,LP 791-18,2960.0,1.893141,very_cool_m,True
1,LP 791-18 b,LP 791-18,2960.0,0.420053,very_cool_m,True
2,TOI-6894 b,TOI-6894,3007.0,16.658991,very_cool_m,True
3,TOI-2267 d,TOI-2267 B,3022.0,0.114210,very_cool_m,True
4,TOI-2267 c,TOI-2267 A,3030.0,0.282000,very_cool_m,True
5,TOI-2267 b,TOI-2267 A,3030.0,0.223000,very_cool_m,True
6,TOI-1080 b,TOI-1080,3065.0,0.368075,very_cool_m,True
7,TOI-1227 b,TOI-1227,3072.0,2.069067,very_cool_m,True
8,TOI-715 b,TOI-715,3075.0,0.448000,very_cool_m,True
9,TOI-6008 b,TOI-6008,3075.0,0.151700,very_cool_m,True


## 7. Real run

When the preview looks good, set `DRY_RUN = False` below and run this cell. This can take time because Lightkurve/MAST searches and downloads real public light curves.

In [ ]:
DRY_RUN = False

# Reload clean original state before the real run, so the dry-run preview does not contaminate the output.
prepare_workspace(uploaded)
meta, existing_targets, existing_names, existing_lc_files = inspect_existing_cache()
candidates = fetch_candidate_targets(existing_names)

updated, added = update_cache_with_new_targets(
    candidates,
    meta,
    existing_targets,
    existing_lc_files
)

print("Real new targets added:", len(added))
pd.DataFrame(added)[["pl_name", "hostname", "st_teff", "pl_trandep", "lightcurve_available", "lightcurve_file"]].head(30)

Existing cache inspection
-------------------------
Targets in JSON:        200
Lightcurve JSON files:  174
Targets with LC flag:   174
Existing target names:  200 unique
Schema:                 exointel-prime-final-merged-cache-v2-cleaned
Querying NASA Exoplanet Archive by stellar temperature bucket...
  very_cool_m   prefer_tess=1     fetched new unique: 126
  very_cool_m   prefer_tess=0     fetched new unique: 44
  cool_k        prefer_tess=1     fetched new unique: 109
  cool_k        prefer_tess=0     fetched new unique: 67
  solar_g       prefer_tess=1     fetched new unique: 79
  solar_g       prefer_tess=0     fetched new unique: 0
  warm_f        prefer_tess=1     fetched new unique: 98
  warm_f        prefer_tess=0     fetched new unique: 51
  hot_a_b       prefer_tess=1     fetched new unique: 3
  hot_a_b       prefer_tess=0     fetched new unique: 11
Total unique candidate targets not in current JSON: 588
Scanning up to 588 candidates.
Goal: add 300 new targets. ADD_TARGETS

Candidates:   0%|          | 0/588 [00:00<?, ?it/s]

  Added with LC: LP 791-18 c -> lp-791-18-c.json (1800 pts)
  Added with LC: LP 791-18 b -> lp-791-18-b.json (1800 pts)
  Added with LC: TOI-6894 b -> toi-6894-b.json (1800 pts)
  Added with LC: TOI-2267 d -> toi-2267-d.json (1800 pts)
  Added with LC: TOI-2267 c -> toi-2267-c.json (1800 pts)
  Added with LC: TOI-2267 b -> toi-2267-b.json (1800 pts)
  Added with LC: TOI-1080 b -> toi-1080-b.json (1800 pts)
  Added with LC: TOI-1227 b -> toi-1227-b.json (1800 pts)
  Added with LC: TOI-715 b -> toi-715-b.json (1800 pts)
  Added with LC: TOI-6008 b -> toi-6008-b.json (1800 pts)
  Added with LC: LHS 3844 b -> lhs-3844-b.json (1800 pts)
  Added with LC: TOI-7166 b -> toi-7166-b.json (1800 pts)
  Added with LC: TOI-6716 b -> toi-6716-b.json (1800 pts)
  Added with LC: TOI-2120 b -> toi-2120-b.json (1800 pts)
  Added with LC: TOI-3884 b -> toi-3884-b.json (1800 pts)
  Added with LC: TOI-7384 b -> toi-7384-b.json (1800 pts)
  Added with LC: TOI-1696 b -> toi-1696-b.json (1800 pts)
  Added with

  Added with LC: TOI-5713 b -> toi-5713-b.json (1800 pts)
  Added with LC: TOI-6002 b -> toi-6002-b.json (1800 pts)
  Added with LC: TOI-6478 b -> toi-6478-b.json (1800 pts)
  Added with LC: TOI-6324 b -> toi-6324-b.json (1800 pts)
  Added with LC: TOI-4552 b -> toi-4552-b.json (1800 pts)
  Added with LC: TOI-4860 b -> toi-4860-b.json (1800 pts)
  Added with LC: TOI-762 A b -> toi-762-a-b.json (1800 pts)
  Added with LC: TOI-1743 b -> toi-1743-b.json (1800 pts)
  Added with LC: TOI-2015 b -> toi-2015-b.json (1800 pts)
  Added with LC: TOI-2096 c -> toi-2096-c.json (1800 pts)
  Added with LC: TOI-2096 b -> toi-2096-b.json (1800 pts)
  Added with LC: LHS 475 b -> lhs-475-b.json (1800 pts)
  Added with LC: TOI-4336 A b -> toi-4336-a-b.json (1800 pts)
  Added with LC: TOI-519 b -> toi-519-b.json (1800 pts)
  Added with LC: TOI-5720 b -> toi-5720-b.json (1800 pts)
  Added with LC: TOI-1224 b -> toi-1224-b.json (1800 pts)
  Added with LC: Gliese 12 b -> gliese-12-b.json (1745 pts)
  Added wi

  Added with LC: TOI-5616 b -> toi-5616-b.json (1800 pts)
  Added with LC: Ross 176 b -> ross-176-b.json (1800 pts)
  Added with LC: TOI-1759 b -> toi-1759-b.json (1800 pts)
  Added with LC: TOI-2005 b -> toi-2005-b.json (1800 pts)
  Added with LC: TOI-1238 b -> toi-1238-b.json (1800 pts)
  Added with LC: TOI-2459 b -> toi-2459-b.json (1800 pts)
  Added with LC: TOI-4898 b -> toi-4898-b.json (1800 pts)
  Added with LC: TOI-1260 d -> toi-1260-d.json (1800 pts)
  Added with LC: TOI-1260 c -> toi-1260-c.json (1800 pts)
  Added with LC: TOI-1260 b -> toi-1260-b.json (1800 pts)
  Added with LC: TOI-5218 b -> toi-5218-b.json (1800 pts)
  Added with LC: TOI-3082 b -> toi-3082-b.json (1800 pts)
  Added with LC: BD-14 3065 b -> bd-14-3065-b.json (1800 pts)
  Added with LC: TOI-1782.01 -> toi-1782-01.json (1800 pts)
  Added with LC: TOI-4153 b -> toi-4153-b.json (1800 pts)
  Added with LC: TOI-615 b -> toi-615-b.json (1800 pts)
  Added with LC: TOI-2443 b -> toi-2443-b.json (1800 pts)
  Added wi

  Added with LC: TOI-1416 b -> toi-1416-b.json (1800 pts)
  Added with LC: TOI-1107 b -> toi-1107-b.json (1800 pts)
  Added with LC: HD 152843 c -> hd-152843-c.json (1800 pts)
  Added with LC: HD 152843 b -> hd-152843-b.json (1800 pts)
  Added with LC: TOI-4551 b -> toi-4551-b.json (1800 pts)
  Added with LC: TOI-205 b -> toi-205-b.json (1800 pts)
  Added with LC: TOI-5153 b -> toi-5153-b.json (1800 pts)
  Added with LC: TOI-677 b -> toi-677-b.json (1800 pts)
  Added with LC: TOI-6707 b -> toi-6707-b.json (1800 pts)


  Added with LC: TOI-1807 b -> toi-1807-b.json (1800 pts)
  Added with LC: TOI-2154 b -> toi-2154-b.json (1800 pts)
  Added with LC: TOI-1333 b -> toi-1333-b.json (1800 pts)
  Added with LC: HIP 113103 b -> hip-113103-b.json (1800 pts)
  Added with LC: TOI-4603 b -> toi-4603-b.json (1800 pts)
  Added with LC: TOI-892 b -> toi-892-b.json (1800 pts)
  Added with LC: TOI-4507 b -> toi-4507-b.json (1800 pts)
  Added with LC: TOI-150.01 -> toi-150-01.json (1800 pts)
  Added with LC: TOI-2046 b -> toi-2046-b.json (1800 pts)
  Added with LC: TOI-628 b -> toi-628-b.json (1800 pts)
  Added with LC: TOI-1898 b -> toi-1898-b.json (1800 pts)
  Added with LC: TOI-333 b -> toi-333-b.json (1800 pts)
  Added with LC: TOI-4010 d -> toi-4010-d.json (1800 pts)
  Added with LC: TOI-5301 b -> toi-5301-b.json (1800 pts)
  Added with LC: TOI-5786 b -> toi-5786-b.json (1800 pts)
  Added with LC: TOI-3568 b -> toi-3568-b.json (1800 pts)
  Added with LC: TOI-4377 b -> toi-4377-b.json (1800 pts)
  Added with LC:

  Added with LC: TOI-5599 b -> toi-5599-b.json (1800 pts)
  Added with LC: TOI-3261 b -> toi-3261-b.json (1800 pts)
  Added with LC: TIC 245076932 b -> tic-245076932-b.json (1800 pts)
  Added with LC: HD 332231 b -> hd-332231-b.json (1800 pts)
  Added with LC: TOI-808 b -> toi-808-b.json (1800 pts)
  Added with LC: HD 221416 b -> hd-221416-b.json (1800 pts)
Real new targets added: 300


,pl_name,hostname,st_teff,pl_trandep,lightcurve_available,lightcurve_file
0,LP 791-18 c,LP 791-18,2960.0,1.893141,True,lp-791-18-c.json
1,LP 791-18 b,LP 791-18,2960.0,0.420053,True,lp-791-18-b.json
2,TOI-6894 b,TOI-6894,3007.0,16.658991,True,toi-6894-b.json
3,TOI-2267 d,TOI-2267 B,3022.0,0.114210,True,toi-2267-d.json
4,TOI-2267 c,TOI-2267 A,3030.0,0.282000,True,toi-2267-c.json
5,TOI-2267 b,TOI-2267 A,3030.0,0.223000,True,toi-2267-b.json
6,TOI-1080 b,TOI-1080,3065.0,0.368075,True,toi-1080-b.json
7,TOI-1227 b,TOI-1227,3072.0,2.069067,True,toi-1227-b.json
8,TOI-715 b,TOI-715,3075.0,0.448000,True,toi-715-b.json
9,TOI-6008 b,TOI-6008,3075.0,0.151700,True,toi-6008-b.json


## 8. Validate and download updated data ZIP

In [ ]:
summary_df = validate_cache(updated)
zip_path = make_output_zip()

from google.colab import files
files.download(str(zip_path))

Updated cache validation
------------------------
Targets:              500
LC flag true:         474
LC files existing:    474
Teff range:           2960.0 K -> 7746.12 K
Lightcurve JSON dir:  /content/exointel_update/data/lightcurves
Output JSON:          /content/exointel_update/data/exoplanets.json
Created: /content/exointel_prime_updated_data.zip
Size: 6.23 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. How to update GitHub

After downloading `exointel_prime_updated_data.zip`:

1. unzip it locally;
2. replace your repo `data/exoplanets.json`;
3. replace/add files inside `data/lightcurves/`;
4. commit and push.

Example:

```bash
git add data/exoplanets.json data/lightcurves
git commit -m "Expand ExoIntel target cache and light curves"
git push
```